In [1]:
# IMPORT LIBRARIES 

import xmltodict
import os
import hashlib

from rdflib import SKOS, RDF, Literal, Graph, Namespace, URIRef

In [2]:
def make_xml_dictionary(file):
    xmlfile = open(file, 'r')
    xml_content = xmlfile.read()
    xml_dictionary = xmltodict.parse(xml_content)
    return xml_dictionary

In [3]:
def define_cbs_variable_thesaurus(var_ns):
    graph.add((URIRef(var_ns), RDF.type, SKOS.ConceptScheme))
    graph.add((URIRef(var_ns), SKOS.prefLabel, Literal("CBS Variables Thesaurus")))
    return graph 

In [4]:
def make_variables_list(xml_dictionary):
    variables_list = xml_dictionary['Dataontwerpversies']['Versie']['Dataontwerp']['Contextvariabelen']['Contextvariabele']
    return variables_list

In [5]:
def make_broader_variable_id(var):
    broader_variable_id = var_ns + var['Variabele']['Id']
    return broader_variable_id

In [6]:
def make_narrower_variable_id(var):
    id_hash = hashlib.sha256((var['Variabele']['Id'] + var['LabelVanDeVariabele']).encode('utf-8')).hexdigest()
    narrower_variable_id = var_ns + id_hash
    return narrower_variable_id

In [7]:
def add_broader_variable_triples(var, broader_variable_id, narrower_variable_id, var_ns):
    graph.add((URIRef(broader_variable_id), RDF.type, SKOS.Concept))
    graph.add((URIRef(broader_variable_id), SKOS.prefLabel, Literal(var['Variabele']['UniekeNaam'], lang='nl')))
    graph.add((URIRef(broader_variable_id), SKOS.definition, Literal(var['Variabele']['Definitie'], lang='nl')))
    graph.add((URIRef(broader_variable_id), SKOS.narrower, URIRef(narrower_variable_id)))
    graph.add((URIRef(broader_variable_id), SKOS.topConceptOf, URIRef(var_ns)))
    graph.add((URIRef(broader_variable_id), SKOS.inScheme, URIRef(var_ns)))
    add_narrower_variable_triples(narrower_variable_id, broader_variable_id)
    return graph

In [8]:
def add_narrower_variable_triples(narrower_variable_id, broader_variable_id):
    graph.add((URIRef(narrower_variable_id), RDF.type, SKOS.Concept))
    graph.add((URIRef(narrower_variable_id), SKOS.prefLabel, Literal(var['LabelVanDeVariabele'], lang='nl')))
    graph.add((URIRef(narrower_variable_id), SKOS.altLabel, Literal(var['VerkorteSchrijfwijzeNaamVariabele'], lang='nl')))
    graph.add((URIRef(narrower_variable_id), SKOS.broader, URIRef(broader_variable_id)))
    return graph

In [9]:
def add_variable_triples(var, var_ns):
    broader_variable_id = make_broader_variable_id(var)
    narrower_variable_id = make_narrower_variable_id(var)
    graph.add((URIRef(var_ns), SKOS.hasTopConcept, URIRef(broader_variable_id)))
    
    add_broader_variable_triples(var, broader_variable_id, narrower_variable_id, var_ns)
    add_narrower_variable_triples(narrower_variable_id, broader_variable_id)
    
    return graph

In [10]:
def write_output_file(graph, ofile):
    with open(ofile, "w") as f:
        f.write(graph.serialize(format="turtle"))

In [11]:
# CONFIGs

# path to directory
directory_path = '../CBS-metadata/ODISSEI_Full_Export_20210924'

# output file name
ofile = "cbs-variables-thesaurus.ttl"

# define namespaces
var_ns = Namespace("https://portal.odissei-data.nl/data/cbs/variableThesaurus/")

In [12]:
graph = Graph()
graph.bind("cbsVar", var_ns)
graph.bind("skos", SKOS)
graph.bind("rdf", RDF)

In [13]:
for file in os.listdir(directory_path):
    
    if file.endswith('.dsc'):
        
        define_cbs_variable_thesaurus(var_ns)

        xml_dictionary = make_xml_dictionary(os.path.join(directory_path, file))
        
        variables_list = make_variables_list(xml_dictionary)
        
        for var in variables_list:
            add_variable_triples(var, var_ns)
        
write_output_file(graph, ofile)